> ### **Building with Sarvam**
>
> **An open teaching kit for the Sarvam AI stack.**
>
> Notebook maintained by **Dr. Bhaveshkumar C. Dharmani** — Founder & AI Mentor, AIVidhya4Sarvam.
PhD (ICT), DA-IICT Gandhinagar · https://www.aividhya.in/ · https://www.aividhya4sarvam.in/ · bhavesh@aividhya.in · https://www.linkedin.com/in/bhaveshdharmani/
>
> Drafted with AI assistance and stress-tested cell by cell in live workshop sessions. Runs against your own Sarvam API key from dashboard.sarvam.ai, with a live ₹ cost meter after every call.
>
> Apache 2.0 · Issues and PRs welcome at github.com/dharmanibc/building-with-sarvam.

<div style="background:#12172E;color:#fff;padding:20px 24px;border-radius:8px">
<div style="color:#FF8A3D;font-size:12px;letter-spacing:2px;font-weight:700">LAB 08 · REAL-TIME VOICE</div>
<div style="font-size:26px;font-weight:700;margin-top:6px">Building a voice agent — and the 800 ms budget</div>
<div style="color:#FFB37A;font-size:14px;margin-top:8px">STT → LLM → TTS loop · latency waterfall · barge-in · telephony · Pipecat/LiveKit</div>
</div>

**Time:** 90 min &nbsp;·&nbsp; **Est. cost:** ≈ ₹12 &nbsp;·&nbsp; **Prereq:** Labs 02, 03, 05, 07

In [1]:
# ── Standard lab header. Run this first in every notebook. ─────────────────
import os, sys, json, time, math, wave, io
from pathlib import Path

# pip install sarvamai python-dotenv

from dotenv import load_dotenv, find_dotenv
# Finds your key without hardcoding anyone's filesystem. Tried in order:
#   1. SARVAM_API_KEY already set in the environment
#   2. the file named by SARVAM_ENV_FILE, if you set that variable
#   3. a .env beside this notebook, or in any parent folder
load_dotenv(os.environ.get("SARVAM_ENV_FILE") or find_dotenv(usecwd=True))

API_KEY = os.environ.get("SARVAM_API_KEY")
assert API_KEY, (
    "SARVAM_API_KEY not found.\\n"
    "Create a .env next to this notebook containing:  SARVAM_API_KEY=sk_...\\n"
    "or point SARVAM_ENV_FILE at an existing env file.\\n"
    "Free key + Rs 1000 credit: https://indus.sarvam.ai/"
)

from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=API_KEY)
DATA = Path("./data"); DATA.mkdir(exist_ok=True)
OUT  = Path("./out");  OUT.mkdir(exist_ok=True)
print("SDK ready ·", sys.version.split()[0])

SDK ready · 3.13.9


In [2]:
# ── The ₹ meter, imported ─────────────────────────────────────────────────
# Lab 00 writes cost_meter.py next to these notebooks. If this import fails,
# run Lab 00 once — it is the only lab that defines the meter.
try:
    from cost_meter import CostMeter, RATES, FREE_CREDIT
except ImportError:
    raise ImportError(
        "cost_meter.py not found.\n"
        "Run 00_Setup_and_the_Cost_Meter.ipynb once — its last section writes "
        "cost_meter.py into this folder, and every other lab imports it from there."
    )

cost = CostMeter()
print(f"cost meter armed · rates dated Aug 2026 · ₹{FREE_CREDIT:.0f} free credit")


cost meter armed · rates dated Aug 2026 · ₹1000 free credit


## The only number that matters

A human conversation tolerates roughly **800 ms** of silence before it feels broken.
Your budget:

```
VAD endpointing   ~200 ms   ─┐
STT finalisation  ~150 ms    │
LLM first token   ~300 ms    ├─  must total < 800 ms
TTS first byte    ~150 ms    │
network             ~50 ms  ─┘
```

**One non-streaming hop blows the entire budget.** That is why Lab 03's REST number
disqualified it for live agents.

> **Teaching note.** Run this lab in **text mode first** (section 2). It exercises the
> exact same pipeline with zero telephony setup and near-zero credit burn. Only move to
> a real phone number once the logic works.

---
## 1 · Measure your own budget before you build

Never trust the docs' latency numbers. Measure on your connection.

In [3]:
import time
from sarvamai.play import save

TURNS = ["नमस्ते, मेरा EMI कब देय है?", "और late fee कितनी है?"]

def measure_llm_first_token(msg, history=None):
    msgs = (history or [{"role": "system", "content": "You are a concise loan agent. One short sentence."}]) + \
           [{"role": "user", "content": msg}]
    t0 = time.perf_counter(); first = None; out = []
    for ch in client.chat.completions(model="sarvam-105b", messages=msgs,
                                      max_tokens=200, reasoning_effort=None, stream=True):
        if ch.choices and getattr(ch.choices[0].delta, "content", None):
            if first is None: first = time.perf_counter() - t0
            out.append(ch.choices[0].delta.content)
    return first, time.perf_counter() - t0, "".join(out)

first, total, text = measure_llm_first_token(TURNS[0])
print(f"LLM first token : {first*1000:>6.0f} ms")
print(f"LLM complete    : {total*1000:>6.0f} ms")
print(f"reply           : {text}")

LLM first token :    305 ms
LLM complete    :    594 ms
reply           : 
नमस्ते, आपकी EMI हर महीने की [तारीख] को देय है।


In [4]:
def measure_tts_first_byte(text):
    t0 = time.perf_counter(); first = None; audio = bytearray()
    # ⚠️ text_to_speech_streaming is the WebSocket client (no .convert()) —
    # HTTP streaming lives on the plain client: .convert_stream(). Same bug as Lab 03.
    for chunk in client.text_to_speech.convert_stream(
            text=text, language_code="hi-IN", model="bulbul:v3", speaker="aditya"):
        if first is None: first = time.perf_counter() - t0
        if isinstance(chunk, (bytes, bytearray)):
            audio.extend(chunk)
    cost.tts(len(text), v3=False)
    return first, time.perf_counter() - t0, bytes(audio)

tf, tt, audio_bytes = measure_tts_first_byte(text)
print(f"TTS first byte  : {tf*1000:>6.0f} ms")
print(f"TTS complete    : {tt*1000:>6.0f} ms   ({len(audio_bytes)} bytes)")

# The streamed response, same way `text` was printed for the LLM call above
tts_out_path = OUT / "tts_first_byte_demo.mp3"     # default output_audio_codec is mp3
tts_out_path.write_bytes(audio_bytes)
print(f"streamed audio  : saved to {tts_out_path}")
from IPython.display import Audio, display
display(Audio(str(tts_out_path)))

print(f"\n── YOUR BUDGET ──")
print(f"LLM first token   {first*1000:>6.0f} ms")
print(f"TTS first byte    {tf*1000:>6.0f} ms")
print(f"subtotal          {(first+tf)*1000:>6.0f} ms   (+ VAD ~200ms + STT ~150ms + net ~50ms)")
print(f"estimated turn    {(first+tf)*1000+400:>6.0f} ms   {'✅ under 800' if (first+tf)*1000+400 < 800 else '⚠️  OVER BUDGET'}")


TTS first byte  :    662 ms
TTS complete    :   1584 ms   (84845 bytes)
streamed audio  : saved to out/tts_first_byte_demo.mp3



── YOUR BUDGET ──
LLM first token      305 ms
TTS first byte       662 ms
subtotal             967 ms   (+ VAD ~200ms + STT ~150ms + net ~50ms)
estimated turn      1367 ms   ⚠️  OVER BUDGET


---
## 2 · The agent loop — text mode

Same pipeline, no telephony. Get the logic right here first.

In [5]:
# ── Tools (reuse from Lab 07) ─────────────────────────────────────────────
ACCOUNTS = {"LN1001": {"name": "Rajesh Kumar", "emi": 12500, "due": "15 अगस्त",
                       "outstanding": 340000, "late_fee_pct": 2}}

def get_account(account_id): return ACCOUNTS.get(account_id, {"error": "not found"})
def get_late_fee(account_id):
    a = ACCOUNTS.get(account_id)
    return {"error": "nf"} if not a else {"pct_per_month": a["late_fee_pct"],
                                          "on_amount": a["emi"]}

REGISTRY = {"get_account": get_account, "get_late_fee": get_late_fee}
TOOLS = [
 {"type":"function","function":{"name":"get_account","description":"Loan account details by ID",
  "parameters":{"type":"object","properties":{"account_id":{"type":"string","description":"e.g. LN1001"}},"required":["account_id"]}}},
 {"type":"function","function":{"name":"get_late_fee","description":"Late payment charge for an account",
  "parameters":{"type":"object","properties":{"account_id":{"type":"string","description":"e.g. LN1001"}},"required":["account_id"]}}},
]

SYSTEM = ("You are a phone agent for an Indian NBFC. Speak like a person on a call: "
          "ONE short sentence, no lists, no markdown. Use tools for any account fact. "
          "Never invent numbers. Always reply in the caller's language. "
          "The caller's account is LN1001.")

In [6]:
class VoiceAgent:
    """The STT -> LLM(+tools) -> TTS loop, with per-turn latency and cost."""

    def __init__(self, language="hi-IN", meter=None):
        self.lang = language
        self.msgs = [{"role": "system", "content": SYSTEM}]
        self.meter = meter
        self.turns = []

    # -- 1. speech in ------------------------------------------------------
    def listen(self, wav_path):
        t0 = time.perf_counter()
        with open(wav_path, "rb") as f:
            r = client.speech_to_text.transcribe(
                file=f, model="saaras:v3", language_code=self.lang, mode="transcribe")
        import wave
        with wave.open(str(wav_path), "rb") as w:
            secs = w.getnframes() / w.getframerate()
        if self.meter: self.meter.stt(secs)
        return r.transcript, time.perf_counter() - t0

    # -- 2. think ----------------------------------------------------------
    def think(self, user_text):
        self.msgs.append({"role": "user", "content": user_text})
        t0 = time.perf_counter(); first = None
        for _ in range(3):
            r = client.chat.completions(model="sarvam-105b", messages=self.msgs,
                                        tools=TOOLS, max_tokens=800, reasoning_effort=None)
            if self.meter: self.meter.llm(r.usage.prompt_tokens, r.usage.completion_tokens)
            m = r.choices[0].message
            calls = getattr(m, "tool_calls", None)
            if not calls:
                self.msgs.append({"role": "assistant", "content": m.content})
                return m.content, time.perf_counter() - t0
            self.msgs.append({"role":"assistant","content":m.content,"tool_calls":
                [{"id":c.id,"type":"function","function":{"name":c.function.name,
                  "arguments":c.function.arguments}} for c in calls]})
            for c in calls:
                out = REGISTRY[c.function.name](**json.loads(c.function.arguments))
                print(f"     🔧 {c.function.name} → {out}")
                self.msgs.append({"role":"tool","tool_call_id":c.id,
                                  "content":json.dumps(out, ensure_ascii=False)})
        return "क्षमा करें, कृपया दोबारा कहिए।", time.perf_counter() - t0

    # -- 3. speak ----------------------------------------------------------
    def speak(self, text, out_path):
        t0 = time.perf_counter()
        a = client.text_to_speech.convert(text=text, language_code=self.lang,
                                          model="bulbul:v3", speaker="priya")
        save(a, str(out_path))
        if self.meter: self.meter.tts(len(text), v3=False)
        return time.perf_counter() - t0

    # -- one full turn -----------------------------------------------------
    def turn(self, wav_in, wav_out):
        heard, t_stt = self.listen(wav_in)
        print(f"  👤 {heard}")
        reply, t_llm = self.think(heard)
        print(f"  🤖 {reply}")
        t_tts = self.speak(reply, wav_out)
        rec = {"stt_ms": t_stt*1000, "llm_ms": t_llm*1000, "tts_ms": t_tts*1000,
               "total_ms": (t_stt+t_llm+t_tts)*1000}
        self.turns.append(rec)
        print(f"  ⏱  stt {rec['stt_ms']:.0f} + llm {rec['llm_ms']:.0f} "
              f"+ tts {rec['tts_ms']:.0f} = {rec['total_ms']:.0f} ms")
        return reply

In [7]:
# Synthesise caller audio so the loop is end-to-end without a phone
CALLER_LINES = ["नमस्ते, मेरा EMI कब देय है?", "और late fee कितनी लगेगी?"]
for i, line in enumerate(CALLER_LINES):
    a = client.text_to_speech.convert(text=line, language_code="hi-IN",
                                      model="bulbul:v3", speaker="rahul")
    save(a, str(DATA / f"caller_{i}.wav")); cost.tts(len(line), v3=False)
print("caller audio ready")

caller audio ready


In [8]:
agent = VoiceAgent(language="hi-IN", meter=cost)
for i in range(len(CALLER_LINES)):
    print(f"\n── TURN {i+1} ──")
    agent.turn(DATA / f"caller_{i}.wav", OUT / f"agent_{i}.wav")

from IPython.display import Audio, display
display(Audio(str(OUT / "agent_0.wav")))


── TURN 1 ──
  👤 नमस्ते, मेरा ईएमआई कब देय है?
     🔧 get_account → {'name': 'Rajesh Kumar', 'emi': 12500, 'due': '15 अगस्त', 'outstanding': 340000, 'late_fee_pct': 2}
  🤖 
नमस्ते राजेश जी, आपकी EMI 15 अगस्त को देय है।
  ⏱  stt 346 + llm 8175 + tts 1831 = 10353 ms

── TURN 2 ──
  👤 और लेट फी कितनी लगेगी?
     🔧 get_late_fee → {'pct_per_month': 2, 'on_amount': 12500}
  🤖 
लेट फी 2% प्रति माह लगेगी, जो ₹250 के बराबर होती है।
  ⏱  stt 592 + llm 1160 + tts 1956 = 3708 ms


In [9]:
# The latency waterfall across the conversation
print(f"{'turn':<6}{'STT':>9}{'LLM':>9}{'TTS':>9}{'TOTAL':>10}")
print("─" * 43)
for i, t in enumerate(agent.turns):
    print(f"{i+1:<6}{t['stt_ms']:>9.0f}{t['llm_ms']:>9.0f}{t['tts_ms']:>9.0f}{t['total_ms']:>10.0f}")
avg = sum(t['total_ms'] for t in agent.turns)/len(agent.turns)
print("─" * 43)
print(f"{'mean':<6}{'':<27}{avg:>10.0f} ms")
print(f"\n⚠️  This is BATCH mode — nothing streams. A real agent streams every hop\n"
      f"    and hides most of this behind the audio already playing.")

turn        STT      LLM      TTS     TOTAL
───────────────────────────────────────────
1           346     8175     1831     10353
2           592     1160     1956      3708
───────────────────────────────────────────
mean                                   7030 ms

⚠️  This is BATCH mode — nothing streams. A real agent streams every hop
    and hides most of this behind the audio already playing.


---
## 3 · Barge-in — the thing that separates a demo from a product

The caller interrupts. You must detect speech, **kill the in-flight TTS stream**,
discard the partial LLM response, and re-enter listening — without losing state.

In [10]:
import asyncio

class BargeInController:
    """Cancellable speak(). This is the core of interruptible voice."""

    def __init__(self):
        self._task = None
        self.interrupted = False

    async def speak(self, text, on_chunk=None):
        self.interrupted = False
        self._task = asyncio.create_task(self._stream(text, on_chunk))
        try:
            await self._task
        except asyncio.CancelledError:
            self.interrupted = True
            print("     ⏹  TTS cancelled mid-utterance")

    async def _stream(self, text, on_chunk):
        # Simulated chunked TTS — replace with the real WebSocket stream
        for i, piece in enumerate(text.split()):
            await asyncio.sleep(0.12)          # ~120 ms per word of audio
            if on_chunk: on_chunk(piece)
        print()

    def interrupt(self):
        if self._task and not self._task.done():
            self._task.cancel()

async def demo_barge_in():
    ctl = BargeInController()
    speaking = asyncio.create_task(
        ctl.speak("आपकी अगली किस्त पंद्रह अगस्त को देय है और राशि बारह हज़ार पाँच सौ रुपये है",
                  on_chunk=lambda w: print(w, end=" ", flush=True)))
    await asyncio.sleep(0.7)                   # caller starts talking after 700 ms
    print("\n     🎤 CALLER INTERRUPTS")
    ctl.interrupt()
    await speaking
    print("     ↩️  re-entering listening state, conversation history intact")

await demo_barge_in()

आपकी अगली किस्त पंद्रह अगस्त 
     🎤 CALLER INTERRUPTS
     ⏹  TTS cancelled mid-utterance
     ↩️  re-entering listening state, conversation history intact


**The four things barge-in must do, in order**

1. **Detect** speech during playback (VAD on the inbound stream)
2. **Cancel** the outbound TTS task immediately — not after the current sentence
3. **Discard** the partial LLM response so it does not get spoken later
4. **Preserve** conversation state — the caller interrupted, they did not reset

Getting 1–3 right but not 4 produces an agent that forgets what it was talking about.

---
## 4 · Telephony — formats and providers

In [11]:
# Phone bridges want 8 kHz mu-law. Generate it correctly.
tel = client.text_to_speech.convert(
    text="आपकी किस्त पंद्रह अगस्त को देय है।",
    language_code="hi-IN", model="bulbul:v3", speaker="ashutosh",
    speech_sample_rate=8000,
    output_audio_codec="mulaw",
)
save(tel, str(OUT / "telephony_out.raw"))
cost.tts(30, v3=False)
print("8 kHz mu-law written — feed this straight to the phone bridge")

8 kHz mu-law written — feed this straight to the phone bridge


| Provider | Best for | Watch out for |
|---|---|---|
| **Exotel** | **India — usually the right answer.** Cheaper, native DLT/TRAI compliance | India-only |
| Twilio | Global reach, excellent docs | Expensive in India, DLT paperwork |
| Vapi | Managed layer, fastest to demo | Less control, another vendor |
| **Sarvam number rental** | **~30 seconds with PAN + Aadhaar** (announced at Epoch) | Verify availability |

> The instant number rental removes the single biggest friction in shipping an Indian
> voice product. Verify it works before you promise it to a workshop room.

---
## 5 · Production frameworks — the sketch

Do not hand-roll the orchestration in production. Use Pipecat (Python-native, easier
to reason about) or LiveKit (WebRTC-native, production-hardened).

In [12]:
PIPECAT_SKELETON = '''
# pip install "pipecat-ai[sarvam,silero]"
from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.task import PipelineTask, PipelineParams
from pipecat.services.sarvam import SarvamSTTService, SarvamTTSService, SarvamLLMService
from pipecat.processors.aggregators.openai_llm_context import OpenAILLMContext
from pipecat.audio.vad.silero import SileroVADAnalyzer

stt = SarvamSTTService(api_key=KEY, model="saaras:v3", language="hi-IN", sample_rate=8000)
llm = SarvamLLMService(api_key=KEY, model="sarvam-105b")
tts = SarvamTTSService(api_key=KEY, model="bulbul:v3", speaker="pooja",
                       sample_rate=8000, codec="mulaw")

llm.register_function("get_account", get_account)
llm.register_function("get_late_fee", get_late_fee)

context = OpenAILLMContext(messages=[{"role": "system", "content": SYSTEM}], tools=TOOLS)

pipeline = Pipeline([
    transport.input(),          # phone / WebRTC in
    stt,                        # streaming speech -> text
    context_aggregator.user(),
    llm,                        # streaming reasoning + tools
    tts,                        # streaming text -> speech
    transport.output(),         # phone / WebRTC out
    context_aggregator.assistant(),
])

task = PipelineTask(pipeline, PipelineParams(
    allow_interruptions=True,          # <- barge-in, handled for you
    enable_metrics=True,               # <- per-hop latency
    vad_analyzer=SileroVADAnalyzer(),  # <- endpointing
))
'''
print(PIPECAT_SKELETON)


# pip install "pipecat-ai[sarvam,silero]"
from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.task import PipelineTask, PipelineParams
from pipecat.services.sarvam import SarvamSTTService, SarvamTTSService, SarvamLLMService
from pipecat.processors.aggregators.openai_llm_context import OpenAILLMContext
from pipecat.audio.vad.silero import SileroVADAnalyzer

stt = SarvamSTTService(api_key=KEY, model="saaras:v3", language="hi-IN", sample_rate=8000)
llm = SarvamLLMService(api_key=KEY, model="sarvam-105b")
tts = SarvamTTSService(api_key=KEY, model="bulbul:v3", speaker="pooja",
                       sample_rate=8000, codec="mulaw")

llm.register_function("get_account", get_account)
llm.register_function("get_late_fee", get_late_fee)

context = OpenAILLMContext(messages=[{"role": "system", "content": SYSTEM}], tools=TOOLS)

pipeline = Pipeline([
    transport.input(),          # phone / WebRTC in
    stt,                        # streaming speech -> text
    context_aggrega

**Pipecat vs LiveKit**

| | Pipecat | LiveKit |
|---|---|---|
| Model | Python pipeline, explicit stages | WebRTC-native agent framework |
| Best for | Learning, custom flows, telephony | Web/app, scale, production hardening |
| Barge-in | `allow_interruptions=True` | Built into the agent loop |
| Sarvam support | First-party plugin + production guide | First-party plugin + production guide |

Start with Pipecat because you can *see* the pipeline. Move to LiveKit when you need
WebRTC at scale.

---
## 6 · The unit economics of this exact agent

In [13]:
def cost_a_call(minutes=3.0, caller_share=0.40, chars_per_min=900,
                turns=8, ctx_tokens=20_000, out_tokens=1_500,
                tts_v3=False, telephony_per_min=0.60, cached_share=0.0):
    stt_sec   = minutes * 60 * caller_share
    tts_chars = minutes * (1 - caller_share) * chars_per_min
    stt  = 30/3600 * stt_sec
    tts  = (30 if tts_v3 else 15)/10_000 * tts_chars
    llm_in  = ctx_tokens * ((1-cached_share)*29.28 + cached_share*10.98) / 1_000_000
    llm_out = out_tokens * 73.20 / 1_000_000
    tel  = telephony_per_min * minutes
    total = stt + tts + llm_in + llm_out + tel
    return {"STT": stt, "TTS": tts, "LLM in": llm_in, "LLM out": llm_out,
            "Telephony": tel, "TOTAL": total}

base = cost_a_call(tts_v3=True)
for k, v in base.items():
    bar = "█" * int(v / base["TOTAL"] * 40)
    print(f"{k:<10} ₹{v:>6.2f}  {bar}")
print(f"\n60% of the bill is TTS. The LLM is {(base['LLM in']+base['LLM out'])/base['TOTAL']:.0%}.")

STT        ₹  0.60  ███
TTS        ₹  4.86  ████████████████████████
LLM in     ₹  0.59  ██
LLM out    ₹  0.11  
Telephony  ₹  1.80  █████████
TOTAL      ₹  7.96  ████████████████████████████████████████

60% of the bill is TTS. The LLM is 9%.


In [14]:
SCENARIOS = [
    ("naive (Bulbul v3)",              dict(tts_v3=True)),
    ("Bulbul v2",                      dict(tts_v3=False)),
    ("v2 + prompt caching",            dict(tts_v3=False, cached_share=0.8)),
    ("v2 + caching + 20% shorter script", dict(tts_v3=False, cached_share=0.8, chars_per_min=720)),
]
for name, kw in SCENARIOS:
    t = cost_a_call(**kw)["TOTAL"]
    print(f"{name:<38} ₹{t:>5.2f}/call   ₹{t*100_000:>10,.0f} per 100k calls")

human = 40
print(f"\nHuman agent (fully loaded)             ₹{human:>5.2f}/call   "
      f"₹{human*100_000:>10,.0f} per 100k calls")

naive (Bulbul v3)                      ₹ 7.96/call   ₹   795,540 per 100k calls
Bulbul v2                              ₹ 5.53/call   ₹   552,540 per 100k calls
v2 + prompt caching                    ₹ 5.23/call   ₹   523,260 per 100k calls
v2 + caching + 20% shorter script      ₹ 4.75/call   ₹   474,660 per 100k calls

Human agent (fully loaded)             ₹40.00/call   ₹ 4,000,000 per 100k calls


In [15]:
cost.report()

TTS          ₹   0.0720  48 chars
TTS          ₹   0.0405  27 chars
TTS          ₹   0.0360  24 chars
STT          ₹   0.0270  3.2s
LLM          ₹   0.0105  273 in / 34 out
LLM          ₹   0.0120  369 in / 17 out
TTS          ₹   0.0690  46 chars
STT          ₹   0.0185  2.2s
LLM          ₹   0.0136  404 in / 24 out
LLM          ₹   0.0152  465 in / 21 out
TTS          ₹   0.0795  53 chars
TTS          ₹   0.0450  30 chars
TOTAL        ₹   0.4388
              (₹1000 free credit → ₹999.56 left)
              Estimated from published rates; actual billing usually lower


0.43878061786848077

---
## ✅ Checkpoint

- [ ] You measured **your own** LLM-first-token and TTS-first-byte latency
- [ ] The text-mode agent completed a 2-turn conversation with a tool call
- [ ] Barge-in cancels TTS mid-utterance and preserves history
- [ ] You produced 8 kHz mu-law output for a phone bridge
- [ ] You can state your cost per call and the three levers that move it

## 🧪 Try this

1. Make every hop **stream** and re-measure. How much of the batch latency disappears?
2. Add a third tool that is slow (2s). Where does the agent need a filler phrase?
3. Rent a number (Exotel or Sarvam) and phone your own agent. This is the moment it becomes real.
4. Run your agent against Lab 07's eval harness. What is the tool-call accuracy over 10 calls?
5. Add a "please hold" utterance when a tool takes >800 ms. Does the call feel better?